In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "ethiopia"
vehicle = "salt"
scenario = "intervention_25_nrv"

In [3]:
# Parameters
location = "india"
vehicle = "rice"
scenario = "intervention"


In [4]:
def aggregate_by_cause_and_scenario(df):
    result = df.groupby(["scenario", "entity", "input_draw", "wealth_quintile"]).value.sum().groupby(["scenario", "entity", "wealth_quintile"]).mean()
    return result[result.index.get_level_values("entity") != "all_causes"]

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylls = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylls.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,lowest,baseline,89,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,lowest,baseline,89,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,second,baseline,89,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,second,baseline,89,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,middle,baseline,89,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
179995,ylls,cause,other_causes,other_causes,95_plus,severe,middle,intervention,13,0,0.0
179996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,fourth,intervention,13,0,0.0
179997,ylls,cause,other_causes,other_causes,95_plus,severe,fourth,intervention,13,0,0.0
179998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,highest,intervention,13,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        100
intervention    100
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"])
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  fourth             191717.387629
                                  highest            176168.539893
                                  lowest             279667.460552
                                  middle             195174.368039
                                  second             225476.079842
intervention  maternal_disorders  fourth             190334.661417
                                  highest            176168.539893
                                  lowest             278310.322816
                                  middle             194685.941078
                                  second             223587.111125
Name: value, dtype: float64

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylds = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,lowest,baseline,89,0,1.237045
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,lowest,baseline,89,0,0.000000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,lowest,baseline,89,0,0.000000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,lowest,baseline,89,0,0.000000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,lowest,baseline,89,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
629995,ylds,cause,pregnancy,parturition,95_plus,severe,highest,intervention,13,0,0.000000
629996,ylds,cause,pregnancy,postpartum,95_plus,severe,highest,intervention,13,0,0.000000
629997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,highest,intervention,13,0,0.000000
629998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,highest,intervention,13,0,0.000000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (pregnancy_ylds[pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])].value == 0).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(lambda df: df[df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])])
pregnancy_ylds_by_scenario

scenario      entity               wealth_quintile
baseline      maternal_hemorrhage  fourth             0.0
                                   highest            0.0
                                   lowest             0.0
                                   middle             0.0
                                   second             0.0
              pregnancy            fourth             0.0
                                   highest            0.0
                                   lowest             0.0
                                   middle             0.0
                                   second             0.0
intervention  maternal_hemorrhage  fourth             0.0
                                   highest            0.0
                                   lowest             0.0
                                   middle             0.0
                                   second             0.0
              pregnancy            fourth             0.0
                     

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(pregnancy_ylds_by_scenario, fill_value=0)
pregnancy_dalys_by_scenario

scenario      entity               wealth_quintile
baseline      maternal_disorders   fourth             191717.387629
                                   highest            176168.539893
                                   lowest             279667.460552
                                   middle             195174.368039
                                   second             225476.079842
              maternal_hemorrhage  fourth                  0.000000
                                   highest                 0.000000
                                   lowest                  0.000000
                                   middle                  0.000000
                                   second                  0.000000
              pregnancy            fourth                  0.000000
                                   highest                 0.000000
                                   lowest                  0.000000
                                   middle                  0.0000

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
    )
else:
    neonatal_ylls = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet").assign(value=0).assign(maternal_scenario=lambda x: x.maternal_scenario.replace('intervention', scenario))
    )

neonatal_ylls = neonatal_ylls.rename(columns={"maternal_scenario": "scenario"})
neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,lowest,baseline,intervention,0,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,second,baseline,intervention,0,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,middle,baseline,intervention,0,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,fourth,baseline,intervention,0,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,highest,baseline,intervention,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
15995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,lowest,baseline,intervention,13,0,46766.401415
15996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,second,baseline,intervention,13,0,33714.095171
15997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,middle,baseline,intervention,13,0,40248.756379
15998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,fourth,baseline,intervention,13,0,13277.047184


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"] == 0).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"]
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario.reset_index().assign(entity="lbwsg").set_index(neonatal_ylls_by_scenario.index.names).value
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   fourth             6.317388e+07
                      highest            6.160879e+07
                      lowest             9.181218e+07
                      middle             6.793668e+07
                      second             7.718123e+07
intervention  lbwsg   fourth             6.280622e+07
                      highest            6.091515e+07
                      lowest             9.100761e+07
                      middle             6.740255e+07
                      second             7.666792e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(path)
    )
else:
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

non_pregnancy_anemia_ylds

,age_start,age_end,sex,wealth_quintile,value,scenario
0,0.0,0.019178,Female,fourth,1059.997610,baseline
1,0.0,0.019178,Female,highest,769.541848,baseline
2,0.0,0.019178,Female,lowest,1801.271863,baseline
3,0.0,0.019178,Female,middle,1244.355229,baseline
4,0.0,0.019178,Female,second,1419.783311,baseline
...,...,...,...,...,...,...
495,95.0,125.000000,Male,fourth,881.484946,intervention
496,95.0,125.000000,Male,highest,1037.502802,intervention
497,95.0,125.000000,Male,lowest,819.728976,intervention
498,95.0,125.000000,Male,middle,857.744221,intervention


In [17]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0"))
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  fourth             3.312747e+06
                      highest            2.946662e+06
                      lowest             4.098008e+06
                      middle             3.612863e+06
                      second             3.684232e+06
intervention  anemia  fourth             3.178329e+06
                      highest            2.778215e+06
                      lowest             3.974837e+06
                      middle             3.484473e+06
                      second             3.566691e+06
Name: value, dtype: float64

In [18]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(path)
    )
else:
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(["scenario", "entity", "wealth_quintile"]).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      ntd     lowest             368777.931735
                      second             325776.815913
                      middle             292471.014886
                      fourth             275785.753509
                      highest            234875.149488
intervention  ntd     fourth             267798.037330
                      highest            227625.844664
                      lowest             358241.208327
                      middle             284779.303424
                      second             317895.044530
Name: value, dtype: float64

In [19]:
dalys_by_scenario = pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0).add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0).add(neural_tube_defect_ylls_by_scenario, fill_value=0)
dalys_by_scenario

scenario      entity               wealth_quintile
baseline      anemia               fourth             3.312747e+06
                                   highest            2.946662e+06
                                   lowest             4.098008e+06
                                   middle             3.612863e+06
                                   second             3.684232e+06
              lbwsg                fourth             6.317388e+07
                                   highest            6.160879e+07
                                   lowest             9.181218e+07
                                   middle             6.793668e+07
                                   second             7.718123e+07
              maternal_disorders   fourth             1.917174e+05
                                   highest            1.761685e+05
                                   lowest             2.796675e+05
                                   middle             1.951744e+05
           

In [20]:
import pathlib

In [21]:
path = f'./results/{location}/{vehicle}/{scenario}/dalys_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)